# Observatorio Inmobiliario CABA — Notebook Maestro End-to-End

Este cuaderno reproduce paso a paso el flujo completo del **workflow de n8n** (`Pipeline Completo Anual y Mensual.json`) directamente en Python, **sin utilizar los actores custom de Apify** (`zonaprop-curator-inference` ni `mlops-annual-trainer`). Únicamente consume el actor scraper de terceros (`memo23~zonaprop-scraper`) y la CKAN API oficial del GCBA.

## Recorrido del Notebook (Pasos 0 a 6)
0. **Setup y Carpetas Dinámicas:** crea `Datasets/`, `etl/output/`, `mlops/artifacts/` y `mlops/figures/` en runtime (el repositorio se transporta limpio sin datasets pesados).
1. **Ingesta Live:** descarga los 4 datasets oficiales de GCBA vía CKAN API y las publicaciones de Zonaprop vía `memo23~zonaprop-scraper`.
2. **ETL, NLP y Enriquecimiento Geoespacial:** unificación monetaria a ARS, extracción NLP (`apto_mascotas`, `score_amenities`) y cruces espaciales (ruido, delitos, escuelas, subtes).
3. **EDA Estadístico y Selección de 10 Features (`VIF < 10`):** pruebas de normalidad/hipótesis, importancia de variables y **Gráficos #1 y #2**.
4. **Modelado (K-Means + Comparativa Baseline vs. XGBoost):** entrenamiento sin *data leakage* (`Pipeline` + `TransformedTargetRegressor`), guardrail de regresión y exportación de artefactos `.joblib`.
5. **Panel de Defensa Ejecutiva (Gráficos #3 a #8):** clustering, comparativa contra baseline, curva de aprendizaje (sesgo-varianza), residuos, auditoría de error por subgrupo y explicabilidad SHAP (Beeswarm + Waterfall).
6. **Inferencia Individual y Batch (`Score 0–100`):** evaluación en vivo para perfiles `FAMILIA` y `SOLTERO_EJECUTIVO` y exportación del CSV listo para el Dashboard Streamlit.

## Paso 0 · Configuración del Entorno y Creación Dinámica de Carpetas

* **Qué hacemos:** detectamos si corremos en local o Google Colab, creamos las carpetas de trabajo en disco (`Datasets/`, `etl/output/`, `mlops/artifacts/`, `mlops/figures/`) y configuramos de forma segura las credenciales y parámetros de ejecución.
* **Por qué importa:** permite transportar el repositorio sin los ~163 MB de datasets crudos; todo se reconstruye de forma reproducible al ejecutar el cuaderno.
* **Qué validar:** que los 5 directorios aparezcan creados y que `APIFY_TOKEN` quede cargado en memoria sin quedar escrito en el archivo `.ipynb`.

In [ ]:
# Paso 0: Detección de entorno (Local vs Google Colab) y configuración de rutas
import getpass
import json
import os
import sys
from pathlib import Path

# Detectamos si estamos en Google Colab para ajustar la raíz del proyecto si hiciera falta
IN_COLAB = "google.colab" in sys.modules
BASE_DIR = Path.cwd()
if not (BASE_DIR / "config" / "settings.py").exists() and (BASE_DIR.parent / "config" / "settings.py").exists():
    BASE_DIR = BASE_DIR.parent

# Agregamos la raíz del repositorio a sys.path para importar los módulos internos del proyecto
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import config.settings as config
import scripts.notebook_helpers as nh

# Creamos dinámicamente las carpetas de datasets y salidas (excluidas de git en .gitignore)
RUNTIME_DIRS = nh.ensure_runtime_dirs(BASE_DIR)

# Parámetros de ingesta Live:
# - APIFY_MODE = "last_run": reutiliza el último run SUCCEEDED de memo23~zonaprop-scraper (no gasta créditos)
# - APIFY_MODE = "run_new": dispara un scraping nuevo con MAX_ITEMS publicaciones (igual que n8n mensual)
APIFY_MODE = os.environ.get("APIFY_MODE", "last_run")
MAX_ITEMS = int(os.environ.get("MAX_ITEMS", "200"))
SKIP_GCBA_IF_EXISTS = True  # Reutiliza archivos GCBA ya descargados en esta sesión

# Lectura segura del token de Apify: primero busca variable de entorno; si falta, solicita por input oculto
APIFY_TOKEN = os.environ.get("APIFY_TOKEN", "").strip()
if not APIFY_TOKEN:
    APIFY_TOKEN = getpass.getpass("Ingresá tu APIFY_TOKEN (no se guarda en el notebook): ").strip()

print(f"[PASO 0] BASE_DIR: {BASE_DIR}")
print(f"[PASO 0] APIFY_MODE={APIFY_MODE} | MAX_ITEMS={MAX_ITEMS} | TOKEN_CARGADO={bool(APIFY_TOKEN)}")
for name, path in RUNTIME_DIRS.items():
    print(f"  - {name}: {path} (existe={path.exists()})")

## Paso 1 · Ingesta Live (CKAN API GCBA + Apify Scraper `memo23~zonaprop-scraper`)

* **Qué hacemos:** consultamos la API oficial CKAN del Gobierno de la Ciudad de Buenos Aires para descargar las 4 capas urbanas (`delitos`, `bocas-subte`, `mapa-ruido`, `establecimientos-educativos`) y obtenemos el JSON crudo de Zonaprop exclusivamente desde el actor scraper de Apify.
* **Por qué importa:** reemplaza la orquestación de n8n sin depender de los actores custom de inferencia o entrenamiento, validando esquemas antes de escribir a disco (`ERR_GCBA_DOWNLOAD`, `ERR_GCBA_SCHEMA`, `ERR_APIFY_SCRAPER_FAILED`).
* **Qué validar:** que los 4 archivos oficiales queden guardados en `Datasets/Originales/` y que `zonaprop_alquileres_caba_1000.json` tenga `>= 1` publicaciones válidas.

In [ ]:
# Paso 1.1: Descarga defensiva de los 4 datasets urbanos desde la CKAN API del GCBA
# Cada archivo se valida en memoria (formato CSV/XLSX y columnas mínimas) antes de escribirse a disco
gcba_report = nh.download_gcba_datasets(
    dest_dir=RUNTIME_DIRS["datasets_originales"],
    skip_if_exists=SKIP_GCBA_IF_EXISTS,
)

# Resolvemos las rutas finales descargadas para inyectarlas luego al pipeline ETL
GCBA_PATHS = {dataset_id: Path(info["path"]) for dataset_id, info in gcba_report.items()}
print("[PASO 1.1] Datasets GCBA listos:")
for dataset_id, info in gcba_report.items():
    print(f"  - {dataset_id}: [{info['tipo']}/{info['razon']}] -> {info['path']}")

# Paso 1.2: Descarga de avisos crudos de Zonaprop usando únicamente el scraper memo23~zonaprop-scraper
ZONAPROP_RAW_PATH = RUNTIME_DIRS["datasets_originales"] / "zonaprop_alquileres_caba_1000.json"
apify_report = nh.fetch_zonaprop_apify(
    token=APIFY_TOKEN,
    output_path=ZONAPROP_RAW_PATH,
    mode=APIFY_MODE,
    max_items=MAX_ITEMS,
)

print(
    f"[PASO 1.2] Zonaprop descargado: [{apify_report['tipo']}] "
    f"mode={apify_report['mode']} | items={apify_report['count']} -> {apify_report['path']}"
)

## Paso 2 · ETL, Ingeniería de Atributos NLP y Enriquecimiento Geoespacial

* **Qué hacemos:** ejecutamos el pipeline de curación (`etl/etl_pipeline.py`) sobre los archivos descargados en el Paso 1: unificación monetaria a ARS (`costo_total_ars`), extracción de expensas y amenities por regex/NLP, limpieza de coordenadas dentro del bounding box de CABA y cruces espaciales (BallTree / GeoPandas) contra ruido, delitos, escuelas y bocas de subte.
* **Por qué importa:** transforma avisos crudos heterogéneos (en ARS y USD, con texto libre) en una tabla analítica limpia y georreferenciada lista para modelar.
* **Qué validar:** cantidad de registros retenidos tras deduplicación/outliers y exportación de los 5 CSVs en `Datasets/Curados/` + `etl/output/dataset_enriquecido_completo.csv`.

In [ ]:
# Paso 2: Ejecución del pipeline ETL completo y exportación de Datasets/Curados/
import pandas as pd
from etl.etl_pipeline import export_all_curated_datasets, run_full_etl

# Pasamos explícitamente las rutas descargadas en el Paso 1 (compatible con .csv o .xlsx)
df_enriched, etl_audit = run_full_etl(
    zonaprop_path=ZONAPROP_RAW_PATH,
    ruido_path=GCBA_PATHS["mapa-ruido"],
    delitos_path=GCBA_PATHS["delitos"],
    escuelas_path=GCBA_PATHS["establecimientos-educativos"],
    subte_path=GCBA_PATHS["bocas-subte"],
)

# Guardamos el dataset enriquecido completo en etl/output/
enriched_csv_path = RUNTIME_DIRS["etl_output"] / "dataset_enriquecido_completo.csv"
df_enriched.to_csv(enriched_csv_path, index=False, encoding="utf-8")

# Exportamos los 5 datasets curados estandarizados en WGS84 en Datasets/Curados/
curated_paths = export_all_curated_datasets(
    df_enriched=df_enriched,
    ruido_path=GCBA_PATHS["mapa-ruido"],
    delitos_path=GCBA_PATHS["delitos"],
    escuelas_path=GCBA_PATHS["establecimientos-educativos"],
    subte_path=GCBA_PATHS["bocas-subte"],
    output_dir=RUNTIME_DIRS["datasets_curados"],
)

print(f"[PASO 2] Dataset enriquecido generado: {len(df_enriched)} filas x {df_enriched.shape[1]} columnas")
print(f"[PASO 2] Guardado en: {enriched_csv_path}")
print("[PASO 2] Resumen de auditoría ETL:", json.dumps(etl_audit, indent=2, ensure_ascii=False)[:600])
df_enriched.head(3)

## Paso 3 · EDA Estadístico, Selección de 10 Features (`VIF < 10`) y Gráficos #1 y #2

* **Qué hacemos:** aplicamos pruebas de normalidad (Shapiro-Wilk), pruebas de hipótesis no paramétricas (Mann-Whitney / Kruskal-Wallis), ranking de importancia (Random Forest + Mutual Information) y poda iterativa de multicolinealidad (`VIF < 10`) para seleccionar las **10 features finales** y exportar `etl/output/producto1_dataset_curado.csv`.
* **Por qué importa:** evita multicolinealidad severa y fundamenta con evidencia estadística la transformación logarítmica `log1p(costo_total_ars)`.
* **Gráficos de Defensa Ejecutiva generados en este paso:**
  * **Gráfico #1 (`01_distribucion_target.png`):** asimetría del target original vs. normalidad tras `log1p`.
  * **Gráfico #2 (`02_mapa_scatter_caba.png`):** distribución geográfica de alquileres en CABA coloreada por precio por $m^2$ (patrón de 3 variables: `lon`, `lat`, `precio_m2`).

In [ ]:
# Paso 3: EDA estadístico, selección de 10 features con VIF < 10 y Gráficos #1 y #2
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from eda.eda_phase2 import (
    CANDIDATE_FEATURES,
    compute_descriptive_stats,
    compute_feature_importance,
    run_hypothesis_tests,
    select_top_10_features,
)

# Configuramos estilo visual limpio y legible para presentación ejecutiva
sns.set_theme(style="whitegrid", context="notebook")
FIG_DIR = RUNTIME_DIRS["mlops_figures"]

# Filtramos las features candidatas disponibles y ejecutamos pruebas estadísticas
available_candidates = [c for c in CANDIDATE_FEATURES if c in df_enriched.columns]
desc_stats = compute_descriptive_stats(df_enriched, available_candidates + [config.TARGET_COL])
hyp_tests = run_hypothesis_tests(df_enriched)
imp_df = compute_feature_importance(df_enriched, available_candidates, config.TARGET_COL)
selected_features, vif_summary = select_top_10_features(
    df_enriched, available_candidates, imp_df, target_col=config.TARGET_COL
)

# Exportamos el Producto 1 (dataset curado con 10 features + target) y reporte JSON
df_curado = df_enriched[selected_features + [config.TARGET_COL]].dropna().copy()
curated_csv_path = RUNTIME_DIRS["etl_output"] / "producto1_dataset_curado.csv"
df_curado.to_csv(curated_csv_path, index=False, encoding="utf-8")

print(f"[PASO 3] Features seleccionadas ({len(selected_features)}):", selected_features)
print(f"[PASO 3] Dataset curado exportado: {curated_csv_path} ({len(df_curado)} filas)")

# --- GRÁFICO #1: Distribución del Target Original vs log1p(Target) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df_enriched[config.TARGET_COL] / 1e6, bins=35, kde=True, ax=axes[0], color="#1f77b4")
axes[0].set_title("Costo Total Mensual (Millones ARS) — Asimetría Positiva")
axes[0].set_xlabel("Costo Total (Millones ARS)")
axes[0].set_ylabel("Frecuencia")

sns.histplot(np.log1p(df_enriched[config.TARGET_COL]), bins=35, kde=True, ax=axes[1], color="#2ca02c")
axes[1].set_title("Transformación log1p(Costo Total) — Estabilización de Varianza")
axes[1].set_xlabel("log1p(Costo Total ARS)")
axes[1].set_ylabel("Frecuencia")
plt.tight_layout()
fig.savefig(FIG_DIR / "01_distribucion_target.png", dpi=140, bbox_inches="tight")
plt.show()

# --- GRÁFICO #2: Mapa Scatter Geográfico de CABA (lon vs lat coloreado por precio/m2) ---
fig, ax = plt.subplots(figsize=(8.5, 6.5))
p_m2 = df_enriched["precio_m2_total_ars"].clip(
    upper=df_enriched["precio_m2_total_ars"].quantile(0.95)
)
sc = ax.scatter(
    df_enriched["longitud"],
    df_enriched["latitud"],
    c=p_m2,
    cmap="viridis",
    s=28,
    alpha=0.75,
    edgecolors="none",
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Precio por m² (ARS, tope P95)")
ax.set_title("Distribución Geoespacial de Alquileres en CABA (Precio/m²)")
ax.set_xlabel("Longitud (WGS84)")
ax.set_ylabel("Latitud (WGS84)")
plt.tight_layout()
fig.savefig(FIG_DIR / "02_mapa_scatter_caba.png", dpi=140, bbox_inches="tight")
plt.show()

## Paso 4 · Modelado: Segmentación K-Means + Comparativa Baseline vs. XGBoost

* **Qué hacemos:**
  1. Entrenamos el modelo no supervisado **K-Means** sobre las 5 variables urbanas (`ruido_db_promedio`, `densidad_delitos_500m`, `distancia_escuela_m`, `lineas_subte_600m`, `score_amenities`) evaluando $k=2..6$.
  2. Comparamos **4 modelos supervisados** encapsulados en `Pipeline` + `TransformedTargetRegressor(log1p)` para evitar *data leakage*: `DummyRegressor` (baseline), `Ridge`, `RandomForestRegressor` y `XGBoost`.
  3. Verificamos el guardrail de regresión (`ERR_METRIC_REGRESSION`) y exportamos los artefactos `.joblib` (`kmeans_model.joblib`, `xgb_price_model.joblib`, `model_metadata.joblib`).
* **Por qué importa:** según el material de SIC (Clase 33 y Clase 36), *ningún modelo demuestra aprendizaje si no supera explícitamente a un baseline ingenuo sobre el mismo split train/test*.
* **Qué validar:** tabla comparativa de `MAE_ars`, `RMSE_ars`, `MAPE_pct` y `R2`, más confirmación de exportación de los 3 archivos `.joblib`.

In [ ]:
# Paso 4: Entrenamiento K-Means + Comparativa Baseline vs Modelos + Exportación .joblib
import joblib
from mlops.train_models import train_kmeans_segmentation, train_xgboost_regressor

# 4.1 Segmentación urbana no supervisada (K-Means) sobre el dataset enriquecido
kmeans_pipe, cluster_labels, kmeans_metrics = train_kmeans_segmentation(
    df_enriched, figures_dir=FIG_DIR
)

# 4.2 Evaluación comparativa sin data leakage: DummyRegressor vs Ridge vs RandomForest vs XGBoost
eval_bundle = nh.evaluate_baseline_vs_models(
    df=df_curado,
    feature_cols=selected_features,
    target_col=config.TARGET_COL,
    min_r2_threshold=0.50,
)
comparison_df = eval_bundle["comparison_df"]
print("[PASO 4] Comparativa Holdout (20%) — Baseline vs Modelos:")
display(comparison_df.round(4))

# 4.3 Entrenamiento final XGBoost (con tuning y 5-Fold CV OOF) + exportación de artefactos .joblib
xgb_model, xgb_metrics, shap_values, X_shap = train_xgboost_regressor(
    df_curado,
    feature_cols=selected_features,
    target_col=config.TARGET_COL,
    figures_dir=FIG_DIR,
)

# Persistimos los 3 artefactos en mlops/artifacts/ listos para el Paso 6 (Inferencia)
ART_DIR = RUNTIME_DIRS["mlops_artifacts"]
joblib.dump(kmeans_pipe, ART_DIR / "kmeans_model.joblib")
joblib.dump(xgb_model, ART_DIR / "xgb_price_model.joblib")
model_metadata = {
    "feature_cols": selected_features,
    "target_col": config.TARGET_COL,
    "cluster_labels": cluster_labels,
    "kmeans_features": kmeans_metrics["features"],
    "holdout_metrics": xgb_metrics["holdout_test_20pct"],
    "kfold_metrics": xgb_metrics["kfold_5_oof_full"],
}
joblib.dump(model_metadata, ART_DIR / "model_metadata.joblib")
print(f"[PASO 4] Artefactos exportados correctamente en: {ART_DIR}")

## Paso 5 · Panel de Defensa Ejecutiva (Gráficos #3 a #8)

* **Qué hacemos:** generamos los 6 gráficos de diagnóstico avanzado inspirados en las Clases 24, 33 y 36 de `apuntes/` para responder las preguntas clave de una defensa técnica y de negocio:
  * **Gráfico #3 (`03_kmeans_elbow_centroides.png`):** elección de $k$ (Elbow + Silhouette) y perfil normalizado de los 5 clusters urbanos.
  * **Gráfico #4 (`04_baseline_vs_modelos.png`):** ganancia real de `XGBoost` frente a `DummyRegressor`, `Ridge` y `RandomForest`.
  * **Gráfico #5 (`05_learning_curve.png`):** diagnóstico de **Sesgo vs. Varianza** (curva de aprendizaje Train vs. 5-Fold CV).
  * **Gráfico #6 (`06_pred_vs_real_residuos.png`):** precisión global (Predicho vs. Real) y análisis de residuos (detección de heterocedasticidad).
  * **Gráfico #7 (`07_error_por_subgrupo.png`):** auditoría *"¿a quién le falla el modelo?"* (MAPE% por barrio y por cuartil de precio).
  * **Gráfico #8 (`08_shap_beeswarm_waterfall.png`):** explicabilidad global (SHAP Beeswarm) y explicación local de 2 casos concretos (1 Oportunidad vs. 1 Sobreprecio).
* **Qué validar:** que los 6 archivos `.png` queden guardados en `mlops/figures/` y sean interpretables a simple vista.

In [ ]:
# Paso 5A: Gráficos #3 (K-Means + Centroides), #4 (Baseline vs Modelos) y #5 (Curva de Aprendizaje)
from sklearn.model_selection import learning_curve
from sklearn.preprocessing import StandardScaler

# --- GRÁFICO #3: K-Means Elbow/Silhouette + Heatmap Normalizado de Centroides ---
k_df = pd.DataFrame(kmeans_metrics["k_evaluation"])
km_feats = kmeans_metrics["features"]
cluster_ids = kmeans_pipe.predict(df_enriched[km_feats])
df_centroids = df_enriched[km_feats].copy()
df_centroids["cluster"] = [cluster_labels.get(int(c), f"Cluster {c}") for c in cluster_ids]
centroids_mean = df_centroids.groupby("cluster")[km_feats].mean()
centroids_z = pd.DataFrame(
    StandardScaler().fit_transform(centroids_mean),
    index=centroids_mean.index,
    columns=centroids_mean.columns,
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax_k = axes[0]
ax_k.plot(k_df["k"], k_df["inertia"], marker="o", color="#1f77b4", label="Inercia (Elbow)")
ax_k.set_xlabel("Número de Clusters (k)")
ax_k.set_ylabel("Inercia", color="#1f77b4")
ax_k2 = ax_k.twinx()
ax_k2.plot(k_df["k"], k_df["silhouette"], marker="s", linestyle="--", color="#d62728", label="Silhouette")
ax_k2.set_ylabel("Silhouette Score", color="#d62728")
ax_k.set_title("Selección de k (Elbow & Silhouette)")

sns.heatmap(centroids_z, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Perfil Normalizado (Z-Score) de los 5 Clusters Urbanos")
plt.tight_layout()
fig.savefig(FIG_DIR / "03_kmeans_elbow_centroides.png", dpi=140, bbox_inches="tight")
plt.show()

# --- GRÁFICO #4: Comparativa Baseline vs Modelos (R2 y MAPE%) ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
palette = ["#7f7f7f", "#1f77b4", "#ff7f0e", "#2ca02c"]
sns.barplot(data=comparison_df, x="model", y="R2", hue="model", palette=palette, legend=False, ax=axes[0])
axes[0].set_title("Comparativa de R² (Holdout 20%) — Mayor es mejor")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=comparison_df, x="model", y="MAPE_pct", hue="model", palette=palette, legend=False, ax=axes[1])
axes[1].set_title("Comparativa de Error Relativo MAPE (%) — Menor es mejor")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
fig.savefig(FIG_DIR / "04_baseline_vs_modelos.png", dpi=140, bbox_inches="tight")
plt.show()

# --- GRÁFICO #5: Curva de Aprendizaje (Diagnóstico de Sesgo vs. Varianza) ---
cv_splits = min(5, max(2, len(df_curado) // 20))
train_sizes, train_scores, val_scores = learning_curve(
    eval_bundle["fitted_models"]["XGBoost"],
    df_curado[selected_features].astype(float),
    df_curado[config.TARGET_COL].astype(float),
    cv=cv_splits,
    scoring="r2",
    train_sizes=np.linspace(0.25, 1.0, 6),
    n_jobs=-1,
)
fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.plot(train_sizes, train_scores.mean(axis=1), "o-", color="#1f77b4", label="R² Entrenamiento")
ax.plot(train_sizes, val_scores.mean(axis=1), "s-", color="#2ca02c", label=f"R² Validación ({cv_splits}-Fold CV)")
ax.fill_between(
    train_sizes,
    val_scores.mean(axis=1) - val_scores.std(axis=1),
    val_scores.mean(axis=1) + val_scores.std(axis=1),
    alpha=0.15,
    color="#2ca02c",
)
ax.set_title("Curva de Aprendizaje XGBoost — Diagnóstico Sesgo-Varianza")
ax.set_xlabel("Cantidad de Propiedades en Entrenamiento")
ax.set_ylabel("R² Score")
ax.legend(loc="lower right")
plt.tight_layout()
fig.savefig(FIG_DIR / "05_learning_curve.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# Paso 5B: Gráficos #6 (Predicho vs Real + Residuos), #7 (Error por Subgrupo) y #8 (SHAP)
import shap

y_test = eval_bundle["y_test"].to_numpy()
y_pred = eval_bundle["xgb_predictions"]
residuals = y_test - y_pred

# --- GRÁFICO #6: Predicho vs Real + Análisis de Residuos ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test / 1e6, y_pred / 1e6, alpha=0.65, color="#1f77b4", edgecolor="white")
lim_max = max(y_test.max(), y_pred.max()) / 1e6
axes[0].plot([0, lim_max], [0, lim_max], "r--", lw=1.8, label="Predicción Perfecta (y = x)")
axes[0].set_title("XGBoost: Precio Predicho vs. Real (Millones ARS)")
axes[0].set_xlabel("Precio Real (Millones ARS)")
axes[0].set_ylabel("Precio Predicho (Millones ARS)")
axes[0].legend()

axes[1].scatter(y_pred / 1e6, residuals / 1e6, alpha=0.65, color="#ff7f0e", edgecolor="white")
axes[1].axhline(0, color="red", linestyle="--", lw=1.5)
axes[1].set_title("Residuos (Real - Predicho) vs. Predicho — Heterocedasticidad")
axes[1].set_xlabel("Precio Predicho (Millones ARS)")
axes[1].set_ylabel("Residuo (Millones ARS)")
plt.tight_layout()
fig.savefig(FIG_DIR / "06_pred_vs_real_residuos.png", dpi=140, bbox_inches="tight")
plt.show()

# --- GRÁFICO #7: Auditoría de Error por Subgrupo (¿A quién le falla el modelo?) ---
df_eval_sub = df_enriched.loc[eval_bundle["X_test"].index, [config.TARGET_COL, "barrio"]].copy()
df_eval_sub["precio_predicho_ars"] = y_pred
subgroup_audit = nh.compute_subgroup_error_audit(df_eval_sub, min_barrio_count=3)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
top_barrios_err = subgroup_audit["by_barrio"].head(10)
sns.barplot(data=top_barrios_err, x="MAPE_pct", y="barrio", color="#d62728", ax=axes[0])
axes[0].set_title("Top Barrios con Mayor Error Relativo (MAPE %)")
axes[0].set_xlabel("MAPE (%)")
axes[0].set_ylabel("Barrio")

sns.barplot(data=subgroup_audit["by_quartile"], x="cuartil_precio", y="MAPE_pct", color="#1f77b4", ax=axes[1])
axes[1].set_title("Error Relativo (MAPE %) por Cuartil de Precio")
axes[1].set_xlabel("Segmento de Precio")
axes[1].set_ylabel("MAPE (%)")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
fig.savefig(FIG_DIR / "07_error_por_subgrupo.png", dpi=140, bbox_inches="tight")
plt.show()

# --- GRÁFICO #8: Explicabilidad SHAP Global (Beeswarm) + Local (Oportunidad vs Sobreprecio) ---
plt.figure(figsize=(10, 5.5))
shap.summary_plot(shap_values, X_shap, show=False)
plt.title("SHAP Beeswarm — Impacto de cada Feature en log1p(Costo Total ARS)")
plt.tight_layout()
plt.savefig(FIG_DIR / "08_shap_beeswarm_waterfall.png", dpi=140, bbox_inches="tight")
plt.show()

# Explicación local (Waterfall) de 2 casos extremos: mayor oportunidad de ahorro vs mayor sobreprecio
preds_full = np.clip(xgb_model.predict(X_shap), 0.0, None)
y_full = df_curado.loc[X_shap.index, config.TARGET_COL].to_numpy()
pct_diff = (y_full - preds_full) / np.clip(preds_full, 1e-6, None)
idx_oportunidad = int(np.argmin(pct_diff))
idx_sobreprecio = int(np.argmax(pct_diff))

print(f"[PASO 5] Caso Oportunidad (idx={idx_oportunidad}): diferencia vs mercado = {pct_diff[idx_oportunidad]*100:.1f}%")
shap.plots.waterfall(shap_values[idx_oportunidad], max_display=8)
print(f"[PASO 5] Caso Sobreprecio (idx={idx_sobreprecio}): diferencia vs mercado = {pct_diff[idx_sobreprecio]*100:+.1f}%")
shap.plots.waterfall(shap_values[idx_sobreprecio], max_display=8)

## Paso 6 · Prueba de Inferencia (Individual + Batch) y Score de Decisión (`0–100`)

* **Qué hacemos:** cargamos los artefactos `.joblib` recién entrenados, probamos una inferencia individual para dos perfiles corporativos (`FAMILIA` y `SOLTERO_EJECUTIVO`) y ejecutamos inferencia batch sobre todo el mercado curado de CABA.
* **Por qué importa:** demuestra el ciclo completo desde dato crudo hasta recomendación accionable y genera `mlops/resultados_inferencia_caba.csv`, el contrato exacto que consume el Dashboard Streamlit.
* **Qué validar:** score de decisión entre `0` y `100`, etiqueta de recomendación (`OPORTUNIDAD`, `PRECIO_JUSTO`, `SOBREPRECIO`) y exportación sin errores del CSV final.

In [ ]:
# Paso 6: Inferencia Individual y Batch con los modelos recién entrenados
from mlops.inference import load_artifacts, predict_batch, predict_property

# Cargamos los artefactos serializados desde mlops/artifacts/
artifacts = load_artifacts(artifacts_dir=RUNTIME_DIRS["mlops_artifacts"])

# Prueba 1: Inferencia individual comparando ambos perfiles sobre una propiedad ejemplo
propiedad_demo = {
    "apto_mascotas": 1,
    "score_amenities": 6.5,
    "m2_totales": 78.0,
    "ruido_db_promedio": 62.0,
    "densidad_delitos_500m": 420.0,
    "distancia_escuela_m": 140.0,
    "lineas_subte_600m": 2,
    "banos": 2,
    "ambientes": 3,
    "cocheras": 1,
    "costo_total_ars": 1_250_000.0,
}

res_familia = predict_property(propiedad_demo, profile="FAMILIA", artifacts=artifacts)
res_ejecutivo = predict_property(propiedad_demo, profile="SOLTERO_EJECUTIVO", artifacts=artifacts)

print("[PASO 6] Resultado Inferencia Individual (FAMILIA):", json.dumps(res_familia, indent=2, ensure_ascii=False))
print("[PASO 6] Resultado Inferencia Individual (SOLTERO_EJECUTIVO):", json.dumps(res_ejecutivo, indent=2, ensure_ascii=False))

# Prueba 2: Inferencia Batch sobre todo el mercado enriquecido de CABA
df_scored = predict_batch(df_enriched, artifacts=artifacts)
output_infer_csv = BASE_DIR / "mlops" / "resultados_inferencia_caba.csv"
df_scored.to_csv(output_infer_csv, index=False, encoding="utf-8")

print(f"[PASO 6] Inferencia Batch completada: {len(df_scored)} propiedades puntuadas -> {output_infer_csv}")
cols_preview = [c for c in ["barrio", "costo_total_ars", "precio_predicho_ars", "pct_sobreprecio", "cluster_label", "score_familia", "score_soltero_ejecutivo"] if c in df_scored.columns]
df_scored[cols_preview].head(10)